# NS09 Tutorial A - Preferential Attachment and the Barabási-Albert Model

**Lecture:** NS09 - Preferential Attachment

**Reading:** Barabási, *Network Science*, Chapter 5

## Learning goals
- Use the lecture definition of the BA rule
  $$
  P(i \leftrightarrow j) = \frac{k_j}{\sum_l k_l}.
  $$
- Understand computationally why **growth alone** is not enough.
- Reproduce the hub-generating effect of **growth + preferential attachment**.
- Connect the mean-field degree-growth rule to the empirical degree distribution.
- Compare linear, sublinear, and superlinear attachment.


In [ ]:
from netsci_utils import *
import pandas as pd

set_seeds()
%matplotlib inline


def polya_urn(n_draws, initial=(1, 1), seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    counts = np.array(initial, dtype=int)
    rows = []
    for step in range(1, n_draws + 1):
        probs = counts / counts.sum()
        colour = rng.choice([0, 1], p=probs)
        counts[colour] += 1
        rows.append({
            't': step,
            'red share': counts[0] / counts.sum(),
            'blue share': counts[1] / counts.sum(),
        })
    return pd.DataFrame(rows)


def ba_from_scratch(n, m, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    G = nx.complete_graph(m + 1)
    attachment_log = []
    for new_node in range(m + 1, n):
        existing = np.array(list(G.nodes()))
        degrees = np.array([G.degree(node) for node in existing], dtype=float)
        probs = degrees / degrees.sum()
        targets = rng.choice(existing, size=m, replace=False, p=probs)
        G.add_node(new_node)
        for target in targets:
            G.add_edge(new_node, int(target))
        attachment_log.append({
            'new node': new_node,
            'targets': tuple(int(target) for target in targets),
        })
    return G, pd.DataFrame(attachment_log)


def random_attachment_growth(n, m, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    G = nx.complete_graph(m + 1)
    for new_node in range(m + 1, n):
        existing = np.array(list(G.nodes()))
        targets = rng.choice(existing, size=m, replace=False)
        G.add_node(new_node)
        for target in targets:
            G.add_edge(new_node, int(target))
    return G


def ba_with_trajectories(n, m, tracked_nodes, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    G = nx.complete_graph(m + 1)
    rows = []
    for t in range(m + 1, n):
        existing = np.array(list(G.nodes()))
        degrees = np.array([G.degree(node) for node in existing], dtype=float)
        probs = degrees / degrees.sum()
        targets = rng.choice(existing, size=m, replace=False, p=probs)
        G.add_node(t)
        for target in targets:
            G.add_edge(t, int(target))

        row = {'time t': t + 1}
        for node in tracked_nodes:
            if node in G:
                row[f'k_{node}(t)'] = G.degree(node)
                row[f'theory_{node}'] = m * np.sqrt((t + 1) / (node + 1))
        rows.append(row)
    return G, pd.DataFrame(rows)


def alpha_attachment_model(n, m, alpha, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    G = nx.complete_graph(m + 1)
    for new_node in range(m + 1, n):
        existing = np.array(list(G.nodes()))
        degrees = np.array([G.degree(node) for node in existing], dtype=float)
        weights = degrees ** alpha
        probs = weights / weights.sum()
        targets = rng.choice(existing, size=m, replace=False, p=probs)
        G.add_node(new_node)
        for target in targets:
            G.add_edge(new_node, int(target))
    return G


def top_degree_table(G, top=10):
    rows = []
    for node, degree in sorted(G.degree, key=lambda item: item[1], reverse=True)[:top]:
        rows.append({'node': node, 'degree': degree})
    return pd.DataFrame(rows)

---
## 1. Polya's urn as a reinforcement analogue

Before building a graph, we simulate a two-colour **Polya urn**. At each step we draw a colour in proportion to its current count and then return it with one extra copy.

This is the lecture intuition behind **rich-get-richer** dynamics.


In [ ]:
urn_history = polya_urn(n_draws=200, initial=(1, 1), seed=RANDOM_SEED)

fig, ax = plt.subplots(figsize=FIGURE_SIZE_SMALL)
ax.plot(urn_history['t'], urn_history['red share'], color=CATEGORY_PALETTE['red'], linewidth=2, label='Red share')
ax.plot(urn_history['t'], urn_history['blue share'], color=CATEGORY_PALETTE['blue'], linewidth=2, label='Blue share')
style_axis(
    ax,
    title="Polya's urn: reinforcement locks in early fluctuations",
    xlabel='Draw number',
    ylabel='Share in the urn',
    legend=True,
)
plt.show()


**Interpretation.**
- Early random advantages do not disappear.
- Reinforcement amplifies them over time.
- BA applies the same logic to network growth, with degree playing the role of the current count.


---
## 2. Build the BA model from scratch on a small graph

The lecture definition has three ingredients:
1. start from a seed graph,
2. add one node at a time,
3. choose each target with probability proportional to current degree.


In [ ]:
ba_small, ba_log = ba_from_scratch(n=15, m=2, seed=RANDOM_SEED)
ba_small_degree = dict(ba_small.degree())
ba_small_pos = nx.spring_layout(ba_small, seed=RANDOM_SEED)

plot_metric(
    ba_small,
    ba_small_degree,
    pos=ba_small_pos,
    with_labels=True,
    colorbar=False,
    min_node_size_px=20,
    max_node_size_px=42,
    title='One small BA graph built from scratch',
)

print('First attachment events:')
print(ba_log.head(8).to_string(index=False))


---
## 3. Is growth alone enough?

We compare two growing models with the same $n$ and $m$:
- **random attachment**: the new node chooses targets uniformly,
- **BA**: the new node chooses targets in proportion to current degree.

This is the lecture control experiment behind the question: does age alone generate hubs?


In [ ]:
n_large = 1500
m_links = 2

random_growth = random_attachment_growth(n_large, m_links, seed=RANDOM_SEED)
ba_large, _ = ba_from_scratch(n_large, m_links, seed=RANDOM_SEED)

comparison_df = pd.DataFrame([
    model_summary_row(random_growth, 'Growth without preferential attachment'),
    model_summary_row(ba_large, 'Barabasi-Albert'),
])
print(comparison_df.round(3).to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=FIGURE_SIZE)
plot_ccdf(
    [degree for _, degree in random_growth.degree()],
    ax=ax,
    label='Growth without PA',
    color=CATEGORY_PALETTE['green'],
    markersize=3,
)
plot_ccdf(
    [degree for _, degree in ba_large.degree()],
    ax=ax,
    label='Barabasi-Albert',
    color=CATEGORY_PALETTE['blue'],
    markersize=3,
)
ax.set_title('Growth alone versus growth + preferential attachment')
ax.legend(frameon=False)
plt.show()

print('Top degrees under random attachment:')
print(top_degree_table(random_growth, top=10).to_string(index=False))
print('\nTop degrees under BA:')
print(top_degree_table(ba_large, top=10).to_string(index=False))


**Interpretation.**
- Growth alone creates an age effect, but not the extreme hubs of BA.
- Preferential attachment is the ingredient that converts cumulative advantage into a broad degree distribution.


---
## 4. Heuristic degree growth for tracked nodes

The lecture mean-field argument says that for a node born at time $t_i$,
$$
k_i(t) \approx m\left(\frac{t}{t_i}\right)^{1/2}.
$$

We track a few nodes over time and compare the simulation to this heuristic formula.


In [ ]:
tracked_nodes = [0, 5, 20, 100]
_, trajectory_df = ba_with_trajectories(n=800, m=2, tracked_nodes=tracked_nodes, seed=RANDOM_SEED)

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
colors = [CATEGORY_PALETTE['blue'], CATEGORY_PALETTE['orange'], CATEGORY_PALETTE['green'], CATEGORY_PALETTE['red']]
for node, color in zip(tracked_nodes, colors):
    observed_col = f'k_{node}(t)'
    theory_col = f'theory_{node}'
    if observed_col in trajectory_df:
        subset = trajectory_df[['time t', observed_col, theory_col]].dropna()
        ax.plot(subset['time t'], subset[observed_col], color=color, linewidth=2, label=f'node {node}: observed')
        ax.plot(subset['time t'], subset[theory_col], color=color, linewidth=1.5, linestyle='--', label=f'node {node}: theory')
style_axis(
    ax,
    title='Tracked BA degree growth versus mean-field prediction',
    xlabel='Network time t',
    ylabel='Degree k_i(t)',
    xscale='log',
    yscale='log',
    legend=True,
)
plt.show()


---
## 5. The three main BA properties in one table

The slides emphasize three structural outcomes:
- a broad degree distribution,
- short paths,
- low clustering.

We compare BA to a real hub-dominated network: **OpenFlights USA**.


In [ ]:
openflights = load_openflights_usa()
compare_real_df = pd.DataFrame([
    model_summary_row(ba_large, 'Barabasi-Albert'),
    model_summary_row(openflights, 'OpenFlights USA'),
])
print(compare_real_df.round(3).to_string(index=False))


In [ ]:
n_values = [200, 500, 1000, 2000]
rows = []
for n in n_values:
    G = nx.barabasi_albert_graph(n, 2, seed=RANDOM_SEED)
    rows.append({
        'n': n,
        'avg path length': nx.average_shortest_path_length(G),
        'avg clustering': nx.average_clustering(G),
        'max degree': max(dict(G.degree()).values()),
    })
scaling_df = pd.DataFrame(rows)
print(scaling_df.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(scaling_df['n'], scaling_df['avg path length'], marker='o', linewidth=2, color=CATEGORY_PALETTE['blue'])
style_axis(axes[0], title='BA path length stays small', xlabel='Network size n', ylabel='Average path length', xscale='log')

axes[1].plot(scaling_df['n'], scaling_df['avg clustering'], marker='o', linewidth=2, color=CATEGORY_PALETTE['orange'])
style_axis(axes[1], title='BA clustering remains low', xlabel='Network size n', ylabel='Average clustering', xscale='log')
plt.show()


**Interpretation.**
- BA reproduces hub emergence and very short paths.
- It does **not** reproduce the high clustering of many real networks, including transportation systems.
- That limitation is exactly why `NS10` introduces extensions.


---
## 6. Why linearity is special

The lecture generalization is
$$
P(i \leftrightarrow j) \propto k_j^{\alpha}.
$$

We compare three regimes:
- **sublinear** $(\alpha < 1)$,
- **linear** $(\alpha = 1)$,
- **superlinear** $(\alpha > 1)$.


In [ ]:
nonlinear_rows = []
alpha_graphs = {}
for alpha, label in [(0.7, 'sublinear'), (1.0, 'linear'), (1.3, 'superlinear')]:
    G = alpha_attachment_model(1500, 2, alpha=alpha, seed=RANDOM_SEED)
    alpha_graphs[label] = G
    degrees = np.array([degree for _, degree in G.degree()])
    nonlinear_rows.append({
        'regime': label,
        'alpha': alpha,
        'max degree': degrees.max(),
        'top-node degree share': degrees.max() / degrees.sum(),
        'kappa': heterogeneity_kappa(G),
    })

nonlinear_df = pd.DataFrame(nonlinear_rows)
print(nonlinear_df.round(4).to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=FIGURE_SIZE)
for label, color in [
    ('sublinear', CATEGORY_PALETTE['green']),
    ('linear', CATEGORY_PALETTE['blue']),
    ('superlinear', CATEGORY_PALETTE['red']),
]:
    plot_ccdf(
        [degree for _, degree in alpha_graphs[label].degree()],
        ax=ax,
        label=label,
        color=color,
        markersize=3,
    )
ax.set_title('Sublinear, linear, and superlinear attachment')
ax.legend(frameon=False)
plt.show()


## Takeaway

- BA explains hub emergence with a **minimal** growth rule.
- Growth alone is not enough; preferential attachment is the mechanism that creates a broad tail.
- The mean-field rule predicts the classical BA scaling
  $$
  p(k) \sim k^{-3}.
  $$
- Linear attachment is the special case that creates hubs without collapsing into winner-takes-all condensation.
